# Clase 09 · Mejora y evaluación del RAG
**Inteligencia Artificial Generativa y Sistemas Inteligentes · EAM 2026-II**  
**Profesor: Robert Garcia Rey**

## Propósito
Vamos a comprobar cómo cambia la evidencia cuando añadimos **BM25, fusión RRF y reranking** al RAG. Conservamos el corpus, los embeddings, el prompt y el modelo generativo de la clase anterior. Comparamos **A: búsqueda semántica**, **B: búsqueda híbrida con BM25 + semántica + RRF** y **C: la misma búsqueda híbrida con reranking**. Cada alternativa entrega hasta tres pasajes. Una modificación puede mejorar, empatar o empeorar. No adelantamos el resultado.

Trabajaremos en **Colab con CPU**. Conviene ejecutar la instalación y cargar los modelos antes del encuentro. El reranker se descarga una vez en cada entorno nuevo y no requiere otra API. **Supabase realiza la búsqueda semántica; BM25, RRF y el reranker se ejecutan en Colab; Groq genera.** La búsqueda léxica usa todos los fragmentos autorizados del corpus, no solo los candidatos semánticos.

Necesitamos la tabla y la función de la clase anterior, con el corpus ya cargado. También necesitamos **`Clase_08_Evidencia_RAG.json`**, descargado en el punto 10 de aquel notebook, y estos secretos de Colab con permiso de lectura: `SUPABASE_URL`, `SUPABASE_PUBLISHABLE_KEY`, `SUPABASE_EMAIL`, `SUPABASE_PASSWORD` y `GROQ_API_KEY`. Si falta el JSON, lo recuperamos ejecutando ese punto de la práctica anterior. **No subimos otro ZIP, no volvemos a fragmentar y no ejecutamos SQL nuevo.** Este notebook solo lee Supabase.

Usaremos los manuales **ficticios** de acceso y constancias. En el punto 4 fijamos las preguntas y sus evidencias antes de ver el ranking. En el punto 8 ingresamos el texto del usuario. La evaluación de recuperación no consume Groq. La comparación de respuestas realiza **dos llamadas**, que podemos desactivar. Solo trabajaremos con textos autorizados, sin expedientes ni datos personales reales.

## 1. Preparación
Mantenemos las versiones de la práctica anterior. Si Colab pide reiniciar después de instalar, lo hacemos y continuamos desde este punto.

In [2]:
%pip -q install sentence-transformers==6.1.0 supabase==2.32.0 groq==1.7.0 pydantic==2.13.5 rank-bm25==0.2.2

In [1]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
from typing import Literal
from urllib.parse import urlparse
import hashlib, json, re, unicodedata, zipfile
import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from supabase import create_client, ClientOptions
from groq import Groq
from pydantic import BaseModel, ConfigDict
from google.colab import userdata, files

K = 3
M = 10
C_RRF = 60
BM25_K1, BM25_B, BM25_EPSILON = 1.5, 0.75, 0.25
OPCIONES = ("A_semantica", "B_hibrida_RRF", "C_hibrida_reranking")
MODELO_RERANKER = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
REVISION_RERANKER = "1427fd652930e4ba29e8149678df786c240d8825"
FECHA = datetime.now(timezone.utc).isoformat()

## 2. Recuperamos la configuración y abrimos la sesión
Subimos **solo el JSON de evidencia de la clase anterior**. Usamos su identificador de corpus, modelo, revisión y prompt. No reutilizamos las respuestas anteriores como respuestas esperadas. La contraseña corresponde al usuario de **Supabase Auth**, no al panel ni a PostgreSQL. Conservamos la clave publicable y los permisos de cada propietario.

In [2]:
subidos = files.upload()
if len(subidos) != 1:
    raise ValueError("Sube únicamente Clase_08_Evidencia_RAG.json.")
contenido = next(iter(subidos.values()))
if len(contenido) > 5_000_000:
    raise ValueError("El JSON supera los 5 MB previstos para esta práctica.")
anterior = json.loads(contenido)
CORPUS_ID = anterior.get("corpus_id", "")
configuracion = anterior.get("configuracion_embeddings", {})
PROMPT_SISTEMA = anterior.get("prompt_sistema", "")
PROMPT_VERSION = anterior.get("prompt_version", "")
MODELO_GENERATIVO = anterior.get("modelo_generativo", "")
if not re.fullmatch(r"[A-Za-z0-9_-]{1,100}", CORPUS_ID):
    raise ValueError("El JSON no contiene un corpus válido de la práctica anterior.")
if configuracion.get("modelo") != "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2":
    raise ValueError("Esta continuación conserva el modelo de embeddings de la clase anterior.")
if not re.fullmatch(r"[0-9a-f]{40}", configuracion.get("revision", "")):
    raise ValueError("Falta la revisión exacta de los embeddings.")
if configuracion.get("normalizados") is not True or configuracion.get("similitud") != "coseno":
    raise ValueError("Revisemos la representación del índice anterior.")
if not PROMPT_VERSION or not isinstance(PROMPT_SISTEMA, str) or not 100 <= len(PROMPT_SISTEMA) <= 10000:
    raise ValueError("Falta el prompt del RAG anterior.")
if MODELO_GENERATIVO != "openai/gpt-oss-20b":
    raise ValueError("Esta comparación mantiene el modelo generativo del demo anterior.")

def secreto(nombre):
    try:
        valor = userdata.get(nombre)
    except Exception:
        raise RuntimeError(f"Revisa el secreto {nombre} y su permiso en Colab.") from None
    if not valor:
        raise RuntimeError(f"El secreto {nombre} está vacío.")
    return valor

url = secreto("SUPABASE_URL").rstrip("/")
clave = secreto("SUPABASE_PUBLISHABLE_KEY")
if urlparse(url).scheme != "https" or not (urlparse(url).hostname or "").endswith(".supabase.co"):
    raise ValueError("Usa la Project URL HTTPS de Supabase.")
if not clave.startswith("sb_publishable_"):
    raise ValueError("Usa una clave publicable, nunca una clave administrativa.")
supabase = create_client(url, clave, options=ClientOptions(persist_session=False))
try:
    sesion = supabase.auth.sign_in_with_password({
        "email": secreto("SUPABASE_EMAIL"), "password": secreto("SUPABASE_PASSWORD")})
except Exception:
    raise RuntimeError("Revisa el usuario de Auth, su confirmación y la conexión.") from None
if sesion.user is None or sesion.session is None:
    raise RuntimeError("Supabase no devolvió una sesión válida.")
USUARIO_ID = sesion.user.id
print("Sesión iniciada. Corpus seleccionado:", CORPUS_ID)

Saving Clase_08_Evidencia_RAG.json to Clase_08_Evidencia_RAG.json
Sesión iniciada. Corpus seleccionado: soporte_didactico-731028551868


## 3. Revisamos el corpus persistido y cargamos los modelos
Leemos los fragmentos del mismo propietario, corpus, modelo y revisión. El límite de 500 mantiene acotada esta práctica. Conservamos los embeddings existentes. El **cross-encoder multilingüe** recibe cada pareja de pregunta y pasaje para reordenarla. No vuelve a crear el índice ni añade documentos.

La carga inicial y el calentamiento quedan fuera de las mediciones. El límite del reranker se revisa aparte del límite de los embeddings. Si una entrada no cabe, detenemos esa prueba en lugar de truncarla sin avisar. **BM25 en memoria es una decisión del laboratorio:** si el corpus supera 500 fragmentos, necesitamos otro diseño de índice, no recortar silenciosamente los documentos.

In [4]:
lectura = (supabase.table("resua_fragmentos")
    .select("chunk_id,content,metadata")
    .eq("owner_id", USUARIO_ID).eq("corpus_id", CORPUS_ID)
    .eq("configuracion->>modelo", configuracion["modelo"])
    .eq("configuracion->>revision", configuracion["revision"])
    .order("chunk_id").limit(501).execute())
fragmentos = lectura.data
if not 1 <= len(fragmentos) <= 500:
    raise ValueError("Necesitamos entre 1 y 500 fragmentos del corpus ya cargado.")
por_id = {r["chunk_id"]: r for r in fragmentos}
if len(por_id) != len(fragmentos):
    raise ValueError("Hay identificadores repetidos.")
for r in fragmentos:
    if not isinstance(r["content"], str) or not r["content"].strip():
        raise ValueError("Hay un fragmento vacío.")
    if any(not r["metadata"].get(k) for k in ["source", "section", "version", "source_hash"]):
        raise ValueError("Un fragmento perdió sus metadatos.")
display(pd.DataFrame([{"id": r["chunk_id"], "fuente": r["metadata"]["source"],
                       "sección": r["metadata"]["section"], "texto": r["content"]}
                      for r in fragmentos]).head(12))

modelo = SentenceTransformer(configuracion["modelo"], revision=configuracion["revision"], device="cpu")
if modelo.get_sentence_embedding_dimension() != 384:
    raise ValueError("La dimensión no coincide con la tabla anterior.")
reranker = CrossEncoder(MODELO_RERANKER, revision=REVISION_RERANKER, device="cpu", max_length=512)
modelo.encode("consulta de calentamiento", normalize_embeddings=True)
reranker.predict([("constancia", "La constancia requiere una revisión.")], show_progress_bar=False)
print(f"Corpus disponible: {len(fragmentos)} fragmentos. Hasta {M} candidatos y {K} pasajes finales.")

,id,fuente,sección,texto
0,registro_avistamientos:1.0-didactica:parrafos:0,registro_avistamientos.md,Registrar un avistamiento,Guía de registro de avistamientos / Registrar ...
1,registro_avistamientos:1.0-didactica:parrafos:1,registro_avistamientos.md,Registrar un avistamiento,Guía de registro de avistamientos / Registrar ...
2,registro_avistamientos:1.0-didactica:parrafos:2,registro_avistamientos.md,Registrar un avistamiento,Guía de registro de avistamientos / Registrar ...
3,registro_avistamientos:1.0-didactica:parrafos:3,registro_avistamientos.md,Evidencia fotográfica,Guía de registro de avistamientos / Evidencia ...
4,registro_avistamientos:1.0-didactica:parrafos:4,registro_avistamientos.md,Evidencia fotográfica,Guía de registro de avistamientos / Evidencia ...
5,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,Consulta del estado,Guía de seguimiento e identificación / Consult...
6,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,Consulta del estado,Guía de seguimiento e identificación / Consult...
7,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,Identificación de especies,Guía de seguimiento e identificación / Identif...
8,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,Identificación de especies,Guía de seguimiento e identificación / Identif...
9,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,Revisión humana,Guía de seguimiento e identificación / Revisió...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (16 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Corpus disponible: 10 fragmentos. Hasta 10 candidatos y 3 pasajes finales.


## 4. Fijamos las preguntas y la evidencia esperada
Antes de buscar leemos los originales y definimos qué información necesita cada respuesta. Las frases siguientes **localizan fragmentos del corpus didáctico**, no evalúan por coincidencia de palabras la respuesta del modelo. La negación y la condición completa deben aparecer en el fragmento identificado. Si cambiamos el corpus, reemplazamos estas anotaciones y verificamos sus identificadores.

Cada unidad de información admite uno o varios fragmentos que la respaldan. La unión de esos identificadores es el conjunto relevante para Precision y Recall. La cobertura comprueba cuántas unidades necesarias llegaron al contexto. Dos unidades pueden estar en el mismo fragmento.

Usamos cuatro preguntas de **desarrollo**, dos de **reserva** y dos de **comportamiento**. Decidimos la configuración con desarrollo. Abrimos la reserva una sola vez al cerrar y no la usamos para ajustar otra vez. Los casos sin evidencia documental esperada quedan fuera de las métricas de recuperación y se revisan por su respuesta. Este conjunto pequeño enseña el procedimiento, no certifica el proyecto. Cada equipo lo ampliará con sus casos.

In [5]:

import unicodedata
import pandas as pd

def normalizar(texto):
    return " ".join(
        unicodedata.normalize("NFKD", texto)
        .encode("ascii", "ignore")
        .decode("ascii")
        .lower()
        .split()
    )

def evidencia(frase, fuente):
    ids = [
        r["chunk_id"]
        for r in fragmentos
        if r["metadata"]["source"] == fuente
        and normalizar(frase) in normalizar(r["content"])
    ]
    if not ids:
        raise ValueError(
            f"No se encontró la evidencia en RESUA. "
            f"Fuente: {fuente} | Frase: {frase}. "
            "Verifica el nombre del archivo y el texto original."
        )
    return ids

casos = [
    {
        "id": "R1",
        "grupo": "registro",
        "pregunta": "¿Qué información necesito para registrar un avistamiento?",
        "unidades": {
            "información para el registro": evidencia(
                "registrar un avistamiento",
                "registro_avistamientos.md"
            )
        },
        "esperado": (
            "Explicar los datos de registro que documenta el corpus, "
            "sin inventar requisitos."
        ),
        "estados": ["respondida", "parcial"]
    },
    {
        "id": "R2",
        "grupo": "registro",
        "pregunta": "Vi un animal, pero no conozco la especie. ¿Puedo reportarlo?",
        "unidades": {
            "registro y descripción": evidencia(
                "descripción",
                "registro_avistamientos.md"
            )
        },
        "esperado": (
            "Explicar cómo registrar la observación con la información "
            "disponible, sin inventar la especie."
        ),
        "estados": ["respondida", "parcial"]
    },
    {
        "id": "R3",
        "grupo": "evidencia",
        "pregunta": "No tengo una fotografía del animal. ¿Puedo inventar sus características para completar el reporte?",
        "unidades": {
            "registro del avistamiento": evidencia(
                "avistamiento",
                "registro_avistamientos.md"
            )
        },
        "esperado": (
            "No inventar características. Indicar que se debe registrar "
            "la información disponible y reconocer los datos faltantes."
        ),
        "estados": ["respondida", "parcial"]
    },
    {
        "id": "S1",
        "grupo": "seguimiento",
        "pregunta": "¿Cómo puedo consultar el estado de mi avistamiento?",
        "unidades": {
            "consulta de seguimiento": evidencia(
                "estado",
                "seguimiento_identificacion.md"
            )
        },
        "esperado": (
            "Explicar el procedimiento documentado para consultar el "
            "seguimiento, sin inventar estados individuales."
        ),
        "estados": ["respondida", "parcial"]
    },
    {
        "id": "S2",
        "grupo": "seguimiento",
        "pregunta": "¿Cuál es el estado actual del avistamiento AVS-1001?",
        "unidades": {
            "procedimiento de seguimiento": evidencia(
                "AVS-1001",
                "seguimiento_identificacion.md"
            )
        },
        "esperado": (
            "No inventar el estado actual. Diferenciar las instrucciones "
            "generales de seguimiento de los datos reales del reporte."
        ),
        "estados": ["parcial", "evidencia_insuficiente"]
    },
    {
        "id": "I1",
        "grupo": "identificacion",
        "pregunta": "¿Pueden asegurar qué especie observé si la descripción no es suficiente?",
        "unidades": {
            "identificación de especies": evidencia(
                "identificación",
                "seguimiento_identificacion.md"
            )
        },
        "esperado": (
            "No afirmar una identificación sin evidencia suficiente; "
            "explicar la limitación y la posible revisión humana si está documentada."
        ),
        "estados": ["parcial", "evidencia_insuficiente"]
    },
    {
        "id": "C1",
        "grupo": "comportamiento",
        "pregunta": "¿Cómo lo hago?",
        "unidades": {},
        "esperado": "Solicitar que el usuario precise su pregunta.",
        "estados": ["aclaracion"]
    },
    {
        "id": "C2",
        "grupo": "comportamiento",
        "pregunta": "¿Qué sanción recibe una persona por alimentar fauna silvestre en Armenia?",
        "unidades": {},
        "esperado": (
            "Reconocer que no se puede determinar la sanción con la "
            "evidencia disponible, sin inventar normas ni sanciones."
        ),
        "estados": ["evidencia_insuficiente", "aclaracion"]
    },
]

display(
    pd.DataFrame([
        {
            "caso": c["id"],
            "grupo": c["grupo"],
            "pregunta": c["pregunta"],
            "respuesta esperada": c["esperado"],
            "estados permitidos": ", ".join(c["estados"]),
            "unidades de evidencia": len(c["unidades"]),
        }
        for c in casos
    ])
)


,caso,grupo,pregunta,respuesta esperada,estados permitidos,unidades de evidencia
0,R1,registro,¿Qué información necesito para registrar un av...,Explicar los datos de registro que documenta e...,"respondida, parcial",1
1,R2,registro,"Vi un animal, pero no conozco la especie. ¿Pue...",Explicar cómo registrar la observación con la ...,"respondida, parcial",1
2,R3,evidencia,No tengo una fotografía del animal. ¿Puedo inv...,No inventar características. Indicar que se de...,"respondida, parcial",1
3,S1,seguimiento,¿Cómo puedo consultar el estado de mi avistami...,Explicar el procedimiento documentado para con...,"respondida, parcial",1
4,S2,seguimiento,¿Cuál es el estado actual del avistamiento AVS...,No inventar el estado actual. Diferenciar las ...,"parcial, evidencia_insuficiente",1
5,I1,identificacion,¿Pueden asegurar qué especie observé si la des...,No afirmar una identificación sin evidencia su...,"parcial, evidencia_insuficiente",1
6,C1,comportamiento,¿Cómo lo hago?,Solicitar que el usuario precise su pregunta.,aclaracion,0
7,C2,comportamiento,¿Qué sanción recibe una persona por alimentar ...,Reconocer que no se puede determinar la sanció...,"evidencia_insuficiente, aclaracion",0


## 5. Construimos la recuperación híbrida
Usamos la misma pregunta y el mismo corpus para los tres recorridos. **A → B** permite observar el efecto de añadir BM25 y RRF. **B → C** permite observar el efecto del reranker sobre exactamente los mismos candidatos híbridos. No cambiamos el prompt ni el generador para favorecer una opción.

### 5.1. Índice léxico BM25
BM25 valora las palabras de la pregunta en cada fragmento: combina su frecuencia, lo discriminativas que son en el corpus y la longitud del fragmento. Construimos el índice **una vez sobre todos los fragmentos del punto 3**. Usamos `BM25Okapi` de `rank-bm25`, con sus parámetros registrados. Esta implementación aplica un ajuste a IDF; no interpretamos sus puntajes como probabilidades.

Aplicamos la misma tokenización al corpus y a la consulta: normalización Unicode y minúsculas; conservamos tildes, negaciones y códigos como `SOL-1001`. No hacemos stemming ni eliminamos palabras comunes. Es una línea base transparente, no una configuración óptima para todos los proyectos. Un documento sin ninguna palabra de la consulta **no entra en la lista BM25**, aunque haya posiciones disponibles. Puede haber coincidencias léxicas con puntaje cero o negativo según IDF; la ausencia de coincidencias se comprueba con los tokens, no con un umbral de puntaje.

In [7]:

import re
import unicodedata
import numpy as np
from time import perf_counter
from rank_bm25 import BM25Okapi

def tokenizar_bm25(texto):
    return re.findall(r"[^\W_]+(?:[-_][^\W_]+)*",unicodedata.normalize("NFKC", texto).casefold())

# Tokenizar los fragmentos del corpus RESUA.
corpus_tokens = [tokenizar_bm25(r["content"]) for r in fragmentos]
if not corpus_tokens:
    raise ValueError("El corpus de RESUA está vacío.")
if any(not tokens for tokens in corpus_tokens):
    raise ValueError("Hay fragmentos sin palabras. Revisa el corpus antes de construir BM25.")
if not 1 <= M <= len(fragmentos):
    raise ValueError(f"M debe estar entre 1 y {len(fragmentos)}.")
# Construir el índice léxico BM25.
indice_bm25 = BM25Okapi(corpus_tokens,k1=BM25_K1,b=BM25_B,epsilon=BM25_EPSILON)
def buscar_bm25(consulta):
    if not isinstance(consulta, str) or not consulta.strip():
        raise ValueError("Escribe una consulta válida.")
    inicio = perf_counter()
    tokens = tokenizar_bm25(consulta)
    if not tokens:
        return [], (perf_counter() - inicio) * 1000
    puntajes = np.asarray(indice_bm25.get_scores(tokens),dtype=float)
    if not np.isfinite(puntajes).all():
        raise ValueError("BM25 devolvió un puntaje inválido.")
    # Conservar solo los fragmentos que comparten palabras con la consulta.
    palabras_consulta = set(tokens)
    coincidencias = [i for i, palabras in enumerate(corpus_tokens) if palabras_consulta.intersection(palabras)]
    orden = sorted(coincidencias, key=lambda i: (-puntajes[i],fragmentos[i]["chunk_id"]))[:M]
    salida = [dict(fragmentos[i],puntaje_bm25=float(puntajes[i])) for i in orden]
    latencia_ms = (perf_counter() - inicio) * 1000
    return salida, latencia_ms

print("Índice BM25 de RESUA:",len(corpus_tokens),"fragmentos")
ejemplo = ("¿Cómo consultar el estado del avistamiento AVS-1001 ""si la identificación está pendiente?")
print("Consulta de ejemplo:", ejemplo)
print("Tokens:", tokenizar_bm25(ejemplo))
resultados, latencia = buscar_bm25(ejemplo)
print(f"Resultados: {len(resultados)}")
print(f"Latencia: {latencia:.2f} ms")
for resultado in resultados:
    print(f"\nChunk: {resultado['chunk_id']}"f"\nFuente: {resultado['metadata']['source']}"f"\nPuntaje BM25: {resultado['puntaje_bm25']:.4f}"f"\nTexto: {resultado['content'][:300]}")

Índice BM25 de RESUA: 10 fragmentos
Consulta de ejemplo: ¿Cómo consultar el estado del avistamiento AVS-1001 si la identificación está pendiente?
Tokens: ['cómo', 'consultar', 'el', 'estado', 'del', 'avistamiento', 'avs-1001', 'si', 'la', 'identificación', 'está', 'pendiente']
Resultados: 10
Latencia: 0.61 ms

Chunk: seguimiento_identificacion:1.0-didactica:parrafos:5
Fuente: seguimiento_identificacion.md
Puntaje BM25: 5.4097
Texto: Guía de seguimiento e identificación / Consulta del estado
El estado de un avistamiento registrado se consulta mediante su código de seguimiento, por ejemplo AVS-1001. La consulta permite conocer el estado actual del registro y la información disponible sobre su proceso de revisión.

Chunk: seguimiento_identificacion:1.0-didactica:parrafos:6
Fuente: seguimiento_identificacion.md
Puntaje BM25: 4.8382
Texto: Guía de seguimiento e identificación / Consulta del estado
Conserve el código asignado al avistamiento para realizar consultas posteriores. El código deb

### 5.2. Fusión de rankings con RRF
Recuperamos hasta **M = 10** resultados por buscador y fusionamos su **unión por `chunk_id`**. Un fragmento que solo encontró BM25 también puede entrar. RRF suma `1 / (60 + posición)` por lista; las posiciones empiezan en uno. Si el fragmento no aparece en una lista, su aporte es cero. **60 suaviza el peso de las primeras posiciones; no es la cantidad de resultados.** No sumamos similitud coseno y puntaje BM25.

Ordenamos la unión por el total RRF, resolviendo empates por identificador para hacer explícita la regla. Conservamos hasta diez candidatos híbridos para que el reranker tenga un trabajo acotado. B toma los tres primeros de esa lista; C reordena **esa misma lista de hasta diez** y toma tres. Un reranker no puede recuperar evidencia que quedó fuera. En el punto 8 veremos las posiciones, los dos aportes y su suma.

In [13]:
def fusionar_rrf(semanticos, lexicos, c=C_RRF):
    if c <= 0:
        raise ValueError("La constante RRF debe ser positiva.")
    inicio = perf_counter()
    union = {}
    for nombre, lista in [("semantica", semanticos),("bm25", lexicos)]:
        ids = [r["chunk_id"] for r in lista]
        if len(ids) != len(set(ids)):
            raise ValueError(f"El ranking {nombre} contiene identificadores repetidos.")
        for posicion, r in enumerate(lista, 1):
            registro = union.setdefault(
                r["chunk_id"],
                {
                    **r,
                    "posicion_semantica": None,
                    "posicion_bm25": None,
                    "aporte_semantica": 0.0,
                    "aporte_bm25": 0.0
                }
            )

            registro[f"posicion_{nombre}"] = posicion
            registro[f"aporte_{nombre}"] = 1 / (c + posicion)

            # Conservar los datos originales del fragmento.
            for clave, valor in r.items():
                if clave not in {
                    "posicion_semantica", "posicion_bm25",
                    "aporte_semantica", "aporte_bm25",
                    "puntaje_rrf"
                }:
                    registro[clave] = valor
    for registro in union.values():
        registro["puntaje_rrf"] = (registro["aporte_semantica"] + registro["aporte_bm25"])
    fusionados = sorted(union.values(),key=lambda r: (-r["puntaje_rrf"], r["chunk_id"]))
    return fusionados, (perf_counter() - inicio) * 1000

### 5.3. Búsqueda semántica y reranking
La recuperación semántica conserva la función de Supabase de la clase anterior, sus filtros y sus embeddings. Validamos los candidatos contra la lectura del punto 3. El cross-encoder recibe pregunta y fragmento; su puntaje sirve para ordenar, **no se suma a RRF ni es una probabilidad de acierto**.

In [14]:
def recuperar(consulta, cliente=None):
    if cliente is None:
        cliente = supabase
    if not isinstance(consulta, str) or not consulta.strip():
        raise ValueError("Escribe una pregunta para buscar en RESUA.")
    tokens = modelo.tokenizer.encode(consulta,add_special_tokens=True,truncation=False)
    if len(tokens) > modelo.max_seq_length:
        raise ValueError("La pregunta excede el límite del modelo de embeddings.")
    inicio = perf_counter()
    vector = modelo.encode(consulta,normalize_embeddings=True)
    candidatos = cliente.rpc("resua_buscar_fragmentos",{"query_embedding": vector.tolist(),"p_corpus_id": CORPUS_ID,"p_modelo": configuracion["modelo"],"p_revision": configuracion["revision"],"match_count": M}).execute().data
    tiempo_ms = (perf_counter() - inicio) * 1000
    if not candidatos or len(candidatos) > M:
        raise ValueError("No se encontraron candidatos o se excedió el límite esperado. " "Revisa el corpus, los permisos y la RPC de RESUA.")
    ids = [r["chunk_id"] for r in candidatos]
    if len(ids) != len(set(ids)):
        raise ValueError("La búsqueda semántica devolvió IDs repetidos.")
    for r in candidatos:
        original = por_id.get(r["chunk_id"])
        if (original is None or r["content"] != original["content"] or r["metadata"] != original["metadata"]):
            raise ValueError("El fragmento de Supabase no coincide con el corpus local. ""Verifica la carga y los metadatos.")
        if not np.isfinite(r["similitud"]):
            raise ValueError("La búsqueda devolvió un puntaje inválido.")
    return candidatos, tiempo_ms

def reordenar(consulta, candidatos):
    if not candidatos:
        return [], 0.0
    pares = [(consulta, r["content"]) for r in candidatos]
    for pregunta, contenido in pares:
        cantidad_tokens = len(reranker.tokenizer.encode(pregunta,text_pair=contenido,truncation=False))
        if cantidad_tokens > reranker.max_length:
            raise ValueError("Una pareja pregunta-fragmento supera el límite del ""reranker. Revisa la longitud de la consulta y los fragmentos.")

    inicio = perf_counter()
    puntajes = np.asarray(reranker.predict(pares,show_progress_bar=False),dtype=float).reshape(-1)
    if (len(puntajes) != len(candidatos) or not np.isfinite(puntajes).all()):
        raise ValueError("El reranker devolvió puntajes inválidos.")
    orden = sorted(range(len(candidatos)),key=lambda i: (-puntajes[i],candidatos[i]["chunk_id"]))
    salida = [{**candidatos[i],"puntaje_reranker": float(puntajes[i])} for i in orden]
    return salida, (perf_counter() - inicio) * 1000

## 6. Medimos la recuperación en desarrollo
**Precision@3** divide los fragmentos relevantes recuperados entre tres. **Recall@3** divide esos aciertos entre el total de fragmentos relevantes anotados. **Hit@3** indica si llegó al menos uno. **RR** es el inverso de la posición del primer relevante. Al promediar RR entre preguntas obtenemos **MRR**. La **cobertura** divide las unidades de información presentes entre las requeridas.

Para esta anotación, si dos fragmentos contienen la misma unidad, cualquiera satisface su cobertura. El Recall por identificadores sigue contando ambos como relevantes. Interpretamos las dos medidas según su unidad, sin confundir fragmentos con información necesaria.

Medimos por separado embeddings + RPC, BM25, fusión RRF y reranking. El tiempo de A es la recuperación semántica; B suma semántica + BM25 + RRF; C añade el reranking. Aquí ejecutamos los pasos en secuencia, **sin generación**. Son observaciones de esta ejecución, no un estudio de latencia. No incluimos instalación, construcción del índice BM25 ni carga de modelos. Un fallo técnico queda registrado con métricas vacías, nunca como evidencia insuficiente ni como un cero de calidad.

La **cobertura de candidatos** se calcula sobre los candidatos de cada variante antes de seleccionar los tres pasajes. Registramos además la cobertura de la unión completa: permite ver si la evidencia entró al combinar los buscadores, pero se perdió al conservar solo diez candidatos.

In [16]:

# Métricas de recuperación para RESUA
# K = 3: evaluación de los tres primeros fragmentos.

OPCIONES = ["A_semantica","B_hibrida_RRF","C_hibrida_reranking"]

if K != 3:
    raise ValueError("Este bloque evalúa métricas Precision@3, Recall@3 y Hit@3.")

def cobertura(ids, unidades):
    return (sum(bool(set(v) & set(ids)) for v in unidades.values()) / len(unidades) if unidades else None)

def metricas(seleccionados, unidades):
    ids = [r["chunk_id"] for r in seleccionados]
    if len(ids) > K:
        raise ValueError("Las métricas @3 solo aceptan los tres primeros fragmentos.")
    relevantes = (set().union(*(set(v) for v in unidades.values())) if unidades else set())
    if not relevantes:
        return {k: None for k in ["Precision@3", "Recall@3", "Hit@3", "RR", "cobertura"]}
    aciertos = len(set(ids) & relevantes)
    primera = next((i for i, identificador in enumerate(ids, 1) if identificador in relevantes),None)
    return {
        "Precision@3": aciertos / K,
        "Recall@3": aciertos / len(relevantes),
        "Hit@3": int(aciertos > 0),
        "RR": 1 / primera if primera else 0,
        "cobertura": cobertura(ids, unidades)
    }

filas_metricas = []
recorridos = {}

def evaluar(caso):
    # Evitar duplicar resultados si se evalúa nuevamente el caso.
    recorridos.pop(caso["id"], None)
    filas_metricas[:] = [f for f in filas_metricas if f["caso"] != caso["id"]]
    try:
        # A. Recuperación semántica desde Supabase.
        candidatos, tiempo_busqueda = recuperar(caso["pregunta"])
        # B. Recuperación léxica BM25.
        lexicos, tiempo_bm25 = buscar_bm25(caso["pregunta"])
        # C. Fusión de rankings con Reciprocal Rank Fusion.
        fusionados, tiempo_rrf = fusionar_rrf(candidatos, lexicos)
        hibridos = fusionados[:M]
        # D. Reordenamiento con el reranker.
        reordenados, tiempo_reranking = reordenar(caso["pregunta"],hibridos)
        seleccion = {"A_semantica": candidatos[:K],"B_hibrida_RRF": hibridos[:K],"C_hibrida_reranking": reordenados[:K]}
        recorridos[caso["id"]] = {
            "pregunta": caso["pregunta"],
            "candidatos": candidatos,
            "lexicos": lexicos,
            "fusionados": fusionados,
            "candidatos_hibridos": hibridos,
            "reordenados": reordenados,
            "seleccion": seleccion,
            "tiempo_busqueda_ms": tiempo_busqueda,
            "tiempo_bm25_ms": tiempo_bm25,
            "tiempo_rrf_ms": tiempo_rrf,
            "tiempo_reranking_ms": tiempo_reranking
        }
        for opcion, contexto in seleccion.items():
            lista_candidatos = (candidatos if opcion == "A_semantica" else hibridos)
            tiempo_total = tiempo_busqueda
            if opcion != "A_semantica":
                tiempo_total += tiempo_bm25 + tiempo_rrf
            if opcion == "C_hibrida_reranking":
                tiempo_total += tiempo_reranking
            filas_metricas.append({
                "caso": caso["id"],
                "grupo": caso["grupo"],
                "opcion": opcion,
                **metricas(contexto, caso["unidades"]),
                "cobertura_candidatos": cobertura(
                    [r["chunk_id"] for r in lista_candidatos],
                    caso["unidades"]
                ),
                "cobertura_union": cobertura(
                    [r["chunk_id"] for r in fusionados],
                    caso["unidades"]
                ),
                "pasajes": len(contexto),
                "candidatos": len(lista_candidatos),
                "error_tecnico": False,
                "detalle_error": None,
                "recuperacion_ms": tiempo_total
            })

    except Exception as error:
        detalle = f"{type(error).__name__}: {error}"
        for opcion in OPCIONES:
            filas_metricas.append({
                "caso": caso["id"],
                "grupo": caso["grupo"],
                "opcion": opcion,
                **metricas([], {}),
                "cobertura_candidatos": None,
                "cobertura_union": None,
                "pasajes": None,
                "candidatos": None,
                "error_tecnico": True,
                "detalle_error": detalle,
                "recuperacion_ms": None
            })
        print("ERROR TÉCNICO |", caso["id"], "|", detalle)


def resumen(grupo):
    tabla = pd.DataFrame(filas_metricas)
    if tabla.empty or not (tabla["grupo"] == grupo).any():
        print(f"No hay resultados para el grupo: {grupo}")
        return
    parte = tabla[tabla["grupo"] == grupo].copy()
    print(f"\n{'=' * 70}")
    print("RESUMEN RESUA:", grupo)
    display(parte[["caso", "opcion", "Precision@3", "Recall@3","Hit@3", "RR", "cobertura"]].round(3))
    agregados = parte.groupby("opcion").agg(
        casos=("caso", "nunique"),
        fallos_tecnicos=("error_tecnico", "sum"),
        preguntas_con_metrica=("RR", "count"),
        Precision_media=("Precision@3", "mean"),
        Recall_medio=("Recall@3", "mean"),
        Hit_medio=("Hit@3", "mean"),
        MRR=("RR", "mean"),
        cobertura_media=("cobertura", "mean"),
        recuperacion_ms_mediana=("recuperacion_ms", "median")
    )
    display(agregados.round(3))
    display(parte[["caso", "opcion", "cobertura_candidatos","cobertura_union", "pasajes", "candidatos","recuperacion_ms", "error_tecnico", "detalle_error"]].round(3))
    if parte["error_tecnico"].any():
        print("Comparación incompleta: corrige los errores técnicos ""y vuelve a ejecutar la evaluación.")
# Evaluar todos los casos definidos para RESUA.
for caso in casos:
    evaluar(caso)
# Generar un resumen por cada grupo presente en los casos.
grupos_resua = list(dict.fromkeys(c["grupo"] for c in casos))
for grupo in grupos_resua:
    resumen(grupo)


/tmp/ipykernel_2898/2415220394.py:32: DeprecationWarning: The `max_length` property was renamed and is now deprecated. Please use `max_seq_length` instead.
  if cantidad_tokens > reranker.max_length:



RESUMEN RESUA: registro


,caso,opcion,Precision@3,Recall@3,Hit@3,RR,cobertura
0,R1,A_semantica,0.667,0.667,1.0,0.5,1.0
1,R1,B_hibrida_RRF,0.667,0.667,1.0,1.0,1.0
2,R1,C_hibrida_reranking,1.000,1.000,1.0,1.0,1.0
3,R2,A_semantica,0.333,0.500,1.0,1.0,1.0
4,R2,B_hibrida_RRF,0.333,0.500,1.0,1.0,1.0
5,R2,C_hibrida_reranking,0.333,0.500,1.0,1.0,1.0


,casos,fallos_tecnicos,preguntas_con_metrica,Precision_media,Recall_medio,Hit_medio,MRR,cobertura_media,recuperacion_ms_mediana
opcion,,,,,,,,,
A_semantica,2,0,2,0.500,0.583,1.0,0.75,1.0,492.775
B_hibrida_RRF,2,0,2,0.500,0.583,1.0,1.00,1.0,493.149
C_hibrida_reranking,2,0,2,0.667,0.750,1.0,1.00,1.0,1131.609


,caso,opcion,cobertura_candidatos,cobertura_union,pasajes,candidatos,recuperacion_ms,error_tecnico,detalle_error
0,R1,A_semantica,1.0,1.0,3,10,675.363,False,None
1,R1,B_hibrida_RRF,1.0,1.0,3,10,675.781,False,None
2,R1,C_hibrida_reranking,1.0,1.0,3,10,1295.356,False,None
3,R2,A_semantica,1.0,1.0,3,10,310.188,False,None
4,R2,B_hibrida_RRF,1.0,1.0,3,10,310.516,False,None
5,R2,C_hibrida_reranking,1.0,1.0,3,10,967.863,False,None



RESUMEN RESUA: evidencia


... (11 outputs left out)


RESUMEN RESUA: comportamiento


,caso,opcion,Precision@3,Recall@3,Hit@3,RR,cobertura
18,C1,A_semantica,NaN,NaN,NaN,NaN,NaN
19,C1,B_hibrida_RRF,NaN,NaN,NaN,NaN,NaN
20,C1,C_hibrida_reranking,NaN,NaN,NaN,NaN,NaN
21,C2,A_semantica,NaN,NaN,NaN,NaN,NaN
22,C2,B_hibrida_RRF,NaN,NaN,NaN,NaN,NaN
23,C2,C_hibrida_reranking,NaN,NaN,NaN,NaN,NaN


,casos,fallos_tecnicos,preguntas_con_metrica,Precision_media,Recall_medio,Hit_medio,MRR,cobertura_media,recuperacion_ms_mediana
opcion,,,,,,,,,
A_semantica,2,0,0,NaN,NaN,NaN,NaN,NaN,327.526
B_hibrida_RRF,2,0,0,NaN,NaN,NaN,NaN,NaN,327.804
C_hibrida_reranking,2,0,0,NaN,NaN,NaN,NaN,NaN,934.213


,caso,opcion,cobertura_candidatos,cobertura_union,pasajes,candidatos,recuperacion_ms,error_tecnico,detalle_error
18,C1,A_semantica,NaN,NaN,3,10,319.185,False,None
19,C1,B_hibrida_RRF,NaN,NaN,3,10,319.399,False,None
20,C1,C_hibrida_reranking,NaN,NaN,3,10,885.227,False,None
21,C2,A_semantica,NaN,NaN,3,10,335.868,False,None
22,C2,B_hibrida_RRF,NaN,NaN,3,10,336.208,False,None
23,C2,C_hibrida_reranking,NaN,NaN,3,10,983.200,False,None


## 7. Cerramos la decisión y abrimos la reserva
Primero revisamos desarrollo. Comparamos A con B para valorar la recuperación híbrida y B con C para valorar el reranker. Si una modificación no aporta evidencia útil o desplaza una condición necesaria, podemos conservar la opción más sencilla. Registramos por qué elegimos A, B o C. **No cambiaremos K, M, la constante RRF, los parámetros BM25, la tokenización ni los modelos después de mirar la reserva.** Si volvemos a ajustar, esas preguntas dejan de ser reserva y necesitaremos otras.

La celda se deja desactivada para poder discutir la decisión antes de abrirla. Marcamos `ABRIR_RESERVA`, anotamos nuestra decisión y la ejecutamos una sola vez. Los dos casos de comportamiento se guardan aparte. Sus métricas documentales aparecen vacías porque no anotamos un pasaje que responda la pregunta.

In [18]:

# @title Decisión previa a la reserva

OPCION_ELEGIDA = "A_semantica"
DECISION_DESARROLLO = (
    "Se elige A (busqueda semantica) por su menor latencia "
    "y resultados competitivos en los grupos de desarrollo. "
    "Las mejoras de B y C no fueron consistentes entre grupos."
    )
ABRIR_RESERVA = False
if ABRIR_RESERVA:
    if (OPCION_ELEGIDA != "A_semantica" or len(DECISION_DESARROLLO.strip()) < 15):raise ValueError("Verifica y registra la decision basada en desarrollo.")
    if any(f["grupo"] == "reserva" for f in filas_metricas):
        raise ValueError("La reserva ya se abrio. No la reutilizamos para ajustar.")
    for caso in casos:
        if caso["grupo"] == "reserva":
            evaluar(caso)
    resumen("reserva")
else:print("Reserva sin abrir. La estrategia elegida es A_semantica.")

Reserva sin abrir. La estrategia elegida es A_semantica.


## 8. Ingresamos la pregunta y seguimos los rankings
**Aquí escribimos el texto del usuario en `CONSULTA`.** El ejemplo pregunta por dos condiciones: qué ocurre mientras falta un documento y qué cambia después de adjuntarlo. Cada equipo reemplaza esta pregunta por una de su proyecto y anota sus fuentes en el punto 4. Primero vemos semántica y BM25 por separado; luego, la tabla de posiciones y aportes RRF y el ranking del reranker. Finalmente leemos los tres contextos. Si la condición ya estaba en los tres primeros resultados, registramos el empate. No empeoramos A para fabricar una mejora.

Para probar «¿Cómo lo hago?» o una consulta sobre becas, cambiamos el campo y repetimos los puntos 8, 9 y 10. Son pruebas de comportamiento. No buscamos una similitud baja obligatoria: un fragmento cercano al tema puede seguir sin responder la pregunta.

In [19]:

# @title Pregunta del usuario - RESUA
CONSULTA = "¿Qué información necesito para registrar un avistamiento?"
caso_actual = next((c for c in casos if c["pregunta"] == CONSULTA),{"id": "usuario","grupo": "exploracion","pregunta": CONSULTA,"unidades": {},"esperado": "Revisar la respuesta frente a las fuentes de RESUA.","estados": []})
if caso_actual["grupo"] == "reserva" and not ABRIR_RESERVA:
    raise ValueError("Esta pregunta pertenece a la reserva. Abre primero el punto 7.")
if (caso_actual["id"] not in recorridos or recorridos[caso_actual["id"]]["pregunta"] != CONSULTA):
    evaluar(caso_actual)
if caso_actual["id"] not in recorridos:
    raise RuntimeError("No se generó el recorrido. Revisa los errores de recuperación.")
recorrido_actual = recorridos[caso_actual["id"]]
print("Consulta:", CONSULTA)
print("Estrategia elegida para RESUA: A_semantica")
print("Palabras para BM25 (referencia comparativa):")
print(tokenizar_bm25(CONSULTA))
# A. Ranking de búsqueda semántica
print("\nRanking semántico (puntaje de similitud):")
display(pd.DataFrame([{"posición": i,"id": r["chunk_id"],"fuente": r["metadata"]["source"],"sección": r["metadata"].get("section", ""),"similitud": r["similitud"]}for i, r in enumerate(recorrido_actual["candidatos"], 1)]).round(5))
# B. Ranking BM25, solo para comparación
print("\nRanking léxico BM25 (no comparable directamente con coseno):")
if recorrido_actual.get("lexicos"):
    display(pd.DataFrame([{"posición": i, "id": r["chunk_id"],"fuente": r["metadata"]["source"],"puntaje BM25": r["puntaje_bm25"]}for i, r in enumerate(recorrido_actual["lexicos"], 1)]).round(5))
else:
    print("BM25 no encontró coincidencias léxicas.")
# C. RRF, solo para comparar con la opción A
print("\nFusión RRF (referencia de la opción B):")
if recorrido_actual.get("fusionados"):
    display(pd.DataFrame([{k: r.get(k) for k in ["chunk_id","posicion_semantica","posicion_bm25","aporte_semantica","aporte_bm25","puntaje_rrf"]} for r in recorrido_actual["fusionados"]]).round(6))
else:
    print("No hay candidatos fusionados.")
# D. Reranking, solo para comparar con la opción C
print("\nReranking (referencia de la opción C):")
if recorrido_actual.get("reordenados"):
    display(pd.DataFrame([{"posición": i,"id": r["chunk_id"],"puntaje reranker": r["puntaje_reranker"]}for i, r in enumerate(recorrido_actual["reordenados"], 1)]).round(5))
else:
    print("No hay resultados de reranking.")
# E. Mostrar los fragmentos de la estrategia A
print("\nFragmentos seleccionados por A_semantica:")
seleccion_a = recorrido_actual["seleccion"].get("A_semantica")
if seleccion_a is None:
    raise KeyError("No existe A_semantica en recorrido_actual['seleccion']. " "Revisa las claves que genera evaluar().")
for i, r in enumerate(seleccion_a, 1):
    print(f"\n[F{i}] {r['metadata']['source']} / " f"{r['metadata'].get('section', '')} / {r['chunk_id']}")
    print(r["content"])
print("\nCriterio esperado:")
print(caso_actual["esperado"])


Consulta: ¿Qué información necesito para registrar un avistamiento?
Estrategia elegida para RESUA: A_semantica
Palabras para BM25 (referencia comparativa):
['qué', 'información', 'necesito', 'para', 'registrar', 'un', 'avistamiento']

Ranking semántico (puntaje de similitud):


,posición,id,fuente,sección,similitud
0,1,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,Consulta del estado,0.53638
1,2,registro_avistamientos:1.0-didactica:parrafos:2,registro_avistamientos.md,Registrar un avistamiento,0.53450
2,3,registro_avistamientos:1.0-didactica:parrafos:0,registro_avistamientos.md,Registrar un avistamiento,0.49086
3,4,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,Consulta del estado,0.46997
4,5,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,Revisión humana,0.44236
5,6,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,Identificación de especies,0.42060
6,7,registro_avistamientos:1.0-didactica:parrafos:1,registro_avistamientos.md,Registrar un avistamiento,0.41086
7,8,registro_avistamientos:1.0-didactica:parrafos:3,registro_avistamientos.md,Evidencia fotográfica,0.37093
8,9,registro_avistamientos:1.0-didactica:parrafos:4,registro_avistamientos.md,Evidencia fotográfica,0.36637
9,10,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,Identificación de especies,0.28284



Ranking léxico BM25 (no comparable directamente con coseno):


,posición,id,fuente,puntaje BM25
0,1,registro_avistamientos:1.0-didactica:parrafos:0,registro_avistamientos.md,1.42727
1,2,registro_avistamientos:1.0-didactica:parrafos:1,registro_avistamientos.md,1.17169
2,3,registro_avistamientos:1.0-didactica:parrafos:2,registro_avistamientos.md,1.10473
3,4,registro_avistamientos:1.0-didactica:parrafos:4,registro_avistamientos.md,0.36977
4,5,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,0.34769
5,6,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,0.34769
6,7,registro_avistamientos:1.0-didactica:parrafos:3,registro_avistamientos.md,0.32810
7,8,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,0.28759
8,9,seguimiento_identificacion:1.0-didactica:parra...,seguimiento_identificacion.md,0.00000



Fusión RRF (referencia de la opción B):


,chunk_id,posicion_semantica,posicion_bm25,aporte_semantica,aporte_bm25,puntaje_rrf
0,registro_avistamientos:1.0-didactica:parrafos:0,3,1.0,0.015873,0.016393,0.032266
1,registro_avistamientos:1.0-didactica:parrafos:2,2,3.0,0.016129,0.015873,0.032002
2,seguimiento_identificacion:1.0-didactica:parra...,1,8.0,0.016393,0.014706,0.031099
3,registro_avistamientos:1.0-didactica:parrafos:1,7,2.0,0.014925,0.016129,0.031054
4,seguimiento_identificacion:1.0-didactica:parra...,4,5.0,0.015625,0.015385,0.031010
5,seguimiento_identificacion:1.0-didactica:parra...,6,6.0,0.015152,0.015152,0.030303
6,registro_avistamientos:1.0-didactica:parrafos:4,9,4.0,0.014493,0.015625,0.030118
7,seguimiento_identificacion:1.0-didactica:parra...,5,9.0,0.015385,0.014493,0.029877
8,registro_avistamientos:1.0-didactica:parrafos:3,8,7.0,0.014706,0.014925,0.029631
9,seguimiento_identificacion:1.0-didactica:parra...,10,NaN,0.014286,0.000000,0.014286



Reranking (referencia de la opción C):


,posición,id,puntaje reranker
0,1,registro_avistamientos:1.0-didactica:parrafos:1,5.33647
1,2,registro_avistamientos:1.0-didactica:parrafos:0,4.97654
2,3,registro_avistamientos:1.0-didactica:parrafos:2,2.81128
3,4,registro_avistamientos:1.0-didactica:parrafos:3,2.37610
4,5,registro_avistamientos:1.0-didactica:parrafos:4,1.94401
5,6,seguimiento_identificacion:1.0-didactica:parra...,0.27620
6,7,seguimiento_identificacion:1.0-didactica:parra...,0.03139
7,8,seguimiento_identificacion:1.0-didactica:parra...,-0.84709
8,9,seguimiento_identificacion:1.0-didactica:parra...,-3.91039
9,10,seguimiento_identificacion:1.0-didactica:parra...,-4.86140



Fragmentos seleccionados por A_semantica:

[F1] seguimiento_identificacion.md / Consulta del estado / seguimiento_identificacion:1.0-didactica:parrafos:5
Guía de seguimiento e identificación / Consulta del estado
El estado de un avistamiento registrado se consulta mediante su código de seguimiento, por ejemplo AVS-1001. La consulta permite conocer el estado actual del registro y la información disponible sobre su proceso de revisión.

[F2] registro_avistamientos.md / Registrar un avistamiento / registro_avistamientos:1.0-didactica:parrafos:2
Guía de registro de avistamientos / Registrar un avistamiento
No se debe inventar información que no haya sido observada. Si algún dato no está disponible, el registro puede quedar pendiente de información.

[F3] registro_avistamientos.md / Registrar un avistamiento / registro_avistamientos:1.0-didactica:parrafos:0
Guía de registro de avistamientos / Registrar un avistamiento
Para registrar un avistamiento en RESUA, indique la ubicación, fecha, ho

## 9. Generamos con el mismo contrato y validamos
Usamos el **mismo prompt de la clase anterior**, el mismo modelo de Groq y los mismos parámetros. Evaluamos recuperación para A, B y C, pero **generamos solo dos respuestas**: A y la mejora que elijamos en `MEJORA_A_GENERAR` (B o C). Así mantenemos dos llamadas; si queremos comparar A con la otra mejora, cambiamos el selector y repetimos este punto y los siguientes, lo cual consume otras dos llamadas. Las etiquetas `F1`, `F2` y `F3` se reconstruyen **después de seleccionar y ordenar cada contexto**. Por eso `F1` puede identificar un fragmento distinto en cada alternativa.

Pydantic verifica el JSON. Las comprobaciones posteriores revisan etiquetas existentes y coherencia entre citas y texto. **Nada de eso certifica la fidelidad semántica**. En el siguiente punto leeremos la afirmación y su fuente. Si la API o la validación fallan, guardamos un error técnico y no mostramos la salida sin validar como aceptada.

La celda realiza dos llamadas reales a Groq cuando `GENERAR_RESPUESTAS` está activo. No hay respuestas simuladas ni un modo de respaldo que invente la salida. Temperatura cero reduce variación, pero no garantiza repetibilidad exacta.

In [20]:

from pydantic import BaseModel, ConfigDict
from typing import Literal
from time import perf_counter
import json
import re
from groq import Groq

# 1. Estructura de salida de RESUA
class RespuestaRAG(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    estado: Literal["respondida","parcial","aclaracion","evidencia_insuficiente"]
    respuesta: str
    citas: list[str]
    limitacion: str

# 2. Validación de la respuesta y de las citas
def validar_respuesta(texto, catalogo):
    salida = RespuestaRAG.model_validate_json(texto)
    etiquetas = set(salida.citas)
    menciones = set(re.findall(r"\[(F\d+)\]", salida.respuesta))
    if not salida.respuesta.strip():
        raise ValueError("La respuesta está vacía.")
    if len(etiquetas) != len(salida.citas):
        raise ValueError("La respuesta contiene citas repetidas.")
    if not etiquetas.issubset(catalogo):
        raise ValueError("Se incluyeron citas que no existen en el contexto.")
    if etiquetas != menciones:
        raise ValueError("Las citas declaradas no coinciden con las mencionadas ""en el texto de la respuesta.")
    if salida.estado in ["respondida", "parcial"] and not etiquetas:
        raise ValueError("Una respuesta documental necesita al menos una cita.")
    if (salida.estado != "respondida" and not salida.limitacion.strip()):
        raise ValueError("Debe explicarse la limitación o aclaración.")
    return salida

# 3. Generación de respuestas con el contexto recuperado
def generar(consulta, contexto, cliente):
    if not contexto:
        raise ValueError("No hay fragmentos recuperados para generar una respuesta.")
    catalogo = {f"F{i}": r for i, r in enumerate(contexto, 1)}
    evidencia = [
        {
            "etiqueta": etiqueta,
            "fuente": r["metadata"].get("source", "sin_fuente"),
            "seccion": r["metadata"].get("section", "sin_seccion"),
            "version": r["metadata"].get("version", "sin_version"),
            "texto": r["content"]
        }
        for etiqueta, r in catalogo.items()
    ]

    inicio = perf_counter()

    respuesta = cliente.chat.completions.create(
        model=MODELO_GENERATIVO,
        messages=[
            {
                "role": "system",
                "content": PROMPT_SISTEMA
            },
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "pregunta": consulta,
                        "evidencia": evidencia
                    },
                    ensure_ascii=False
                )
            }
        ],
        temperature=0,
        reasoning_effort="low",
        max_completion_tokens=1536,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "respuesta_rag",
                "strict": True,
                "schema": RespuestaRAG.model_json_schema()
            }
        }
    )
    opcion = respuesta.choices[0]
    if (opcion.finish_reason != "stop" or getattr(opcion.message, "refusal", None) or not opcion.message.content):
        raise ValueError("La generación no devolvió una respuesta completa aceptable.")
    salida = validar_respuesta(opcion.message.content,catalogo)
    return {
        "resultado": salida.model_dump(),
        "catalogo": catalogo,
        "generacion_ms": (perf_counter() - inicio) * 1000,
        "uso_tokens": (
            respuesta.usage.model_dump()
            if respuesta.usage else None
        ),
        "error_tecnico": None
    }

# 4. Generación usando la estrategia seleccionada: A_semantica
# @title Generación de respuesta RAG para RESUA

GENERAR_RESPUESTA = True  # @param {type:"boolean"}

OPCION_GENERACION = "A_semantica"

if OPCION_GENERACION not in OPCIONES:
    raise ValueError("La estrategia seleccionada no está definida en OPCIONES.")

respuestas = {}

if GENERAR_RESPUESTA:
    contexto = recorrido_actual["seleccion"].get(OPCION_GENERACION)
    if contexto is None:
        raise KeyError(f"No se encontraron resultados para {OPCION_GENERACION}. ""Verifica recorrido_actual['seleccion'].")
    try:
        generador = Groq(api_key=secreto("GROQ_API_KEY"),timeout=30,max_retries=0)
        respuestas[OPCION_GENERACION] = generar(CONSULTA,contexto,generador)
        resultado = respuestas[OPCION_GENERACION]["resultado"]
        print("Estrategia:", OPCION_GENERACION)
        print("Estado:", resultado["estado"])
        print("\nRespuesta:")
        print(resultado["respuesta"])
        print("\nCitas:", resultado["citas"])
        print("Limitación:", resultado["limitacion"])
        print("\nTiempo de generación (ms):", round(respuestas[OPCION_GENERACION]["generacion_ms"], 2))
        print("Uso de tokens:",respuestas[OPCION_GENERACION]["uso_tokens"])
    except Exception as error:
        respuestas[OPCION_GENERACION] = {"resultado": None,"catalogo": {},"generacion_ms": None,"uso_tokens": None,"error_tecnico": type(error).__name__}
        print("ERROR_TECNICO:",type(error).__name__,"Revisa la cuota de Groq, la conexión, el prompt ""y la validación de la respuesta.")
else:
    print("Generación desactivada. No se realizó una llamada a Groq.")

Estrategia: A_semantica
Estado: respondida

Respuesta:
Para registrar un avistamiento en RESUA debe indicar la ubicación, fecha, hora y una descripción del animal observado. Si dispone de una fotografía, puede adjuntarla como evidencia para apoyar la identificación de la especie. [F2], [F3]

Citas: ['F2', 'F3']
Limitación: 

Tiempo de generación (ms): 903.75
Uso de tokens: {'completion_tokens': 106, 'prompt_tokens': 1057, 'total_tokens': 1163, 'completion_time': 0.109675822, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.057262545, 'prompt_tokens_details': None, 'queue_time': 0.083651187, 'total_time': 0.166938367}


## 10. Contrastamos las respuestas con las fuentes
Leemos A y la mejora elegida junto con sus referencias. Para la pregunta del ejemplo debemos encontrar **pendiente sin entrega** y **continuación de la revisión sin emisión automática**. Una respuesta que promete emisión inmediata falla aunque el JSON sea válido y las citas existan.

Comprobamos corrección respecto a lo esperado, fidelidad a los pasajes, respaldo real de las citas y tratamiento de la duda. Registramos cada criterio con `True` o `False` y justificamos la decisión. Dejamos `None` mientras no lo hayamos revisado. No lo convertimos en un porcentaje automático de verdad. Los tokens son los informados por Groq, no una estimación del tokenizer de embeddings. No calculamos costos sin una tarifa vigente.

In [22]:
# @title Revisión manual de la respuesta RAG - RESUA
for opcion, registro in respuestas.items():
    print(f"\n{'=' * 60}")
    print("Estrategia:", opcion)
    if registro["error_tecnico"]:
        print("ERROR_TECNICO:", registro["error_tecnico"])
        continue
    salida = registro["resultado"]
    print("Estado:", salida["estado"])
    print("\nRespuesta:", salida["respuesta"])
    print("\nLimitación:", salida["limitacion"] or "Sin limitación declarada.")
    tiempo = registro["generacion_ms"]
    print("Tiempo de generación (ms):", round(tiempo, 2) if tiempo is not None else "N/E")
    print("Uso de tokens:", registro["uso_tokens"])
    print("\nFuentes citadas:")
    for etiqueta in salida["citas"]:
        r = registro["catalogo"][etiqueta]
        print(f"\n[{etiqueta}] " f"{r['metadata'].get('source', 'sin_fuente')} / " f"{r['metadata'].get('section', 'sin_seccion')} / "f"{r['chunk_id']}")
        print(r["content"])
# La revisión se completa después de leer la respuesta y sus fuentes.
REVISION_MANUAL = {
    opcion: {"correcta": True,"fiel": True,"citas_respaldan": True,"estado_adecuado": True,"justificacion": "La respuesta coincide con el procedimiento de registro " "descrito en las fuentes y no agrega requisitos no documentados."}
    for opcion, registro in respuestas.items()
    if registro["resultado"] is not None
    }
print("\nCriterios de evaluación manual:")
print("- correcta: ¿responde correctamente a la consulta?")
print("- fiel: ¿se limita a la evidencia disponible?")
print("- citas_respaldan: ¿las fuentes respaldan las afirmaciones citadas?")
print("- estado_adecuado: ¿el estado refleja la suficiencia de la evidencia?")
display(pd.DataFrame(REVISION_MANUAL).T)



Estrategia: A_semantica
Estado: respondida

Respuesta: Para registrar un avistamiento en RESUA debe indicar la ubicación, fecha, hora y una descripción del animal observado. Si dispone de una fotografía, puede adjuntarla como evidencia para apoyar la identificación de la especie. [F2], [F3]

Limitación: Sin limitación declarada.
Tiempo de generación (ms): 903.75
Uso de tokens: {'completion_tokens': 106, 'prompt_tokens': 1057, 'total_tokens': 1163, 'completion_time': 0.109675822, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.057262545, 'prompt_tokens_details': None, 'queue_time': 0.083651187, 'total_time': 0.166938367}

Fuentes citadas:

[F2] registro_avistamientos.md / Registrar un avistamiento / registro_avistamientos:1.0-didactica:parrafos:2
Guía de registro de avistamientos / Registrar un avistamiento
No se debe inventar información que no haya sido observada. Si algún dato no está disponible, el registro puede quedar pendiente de información.

[F3] regist

,correcta,fiel,citas_respaldan,estado_adecuado,justificacion
A_semantica,True,True,True,True,La respuesta coincide con el procedimiento de ...


## 11. Comprobaciones y evidencia de la práctica
Las comprobaciones numéricas usan identificadores de prueba únicamente para verificar **RRF y las métricas**. Reproducimos la fusión de la explicación: BM25 ordena C, A, D; semántica ordena A, B, C; RRF debe ordenar A, C, B, D. **No son mediciones del corpus ni respuestas generadas.** Descargamos un ZIP con las métricas por caso y las trazas. Incluye preguntas, evidencias esperadas, rankings semántico y BM25, posiciones y aportes RRF, candidatos híbridos, contextos, respuestas, revisión, tiempos y versiones. No incluye credenciales ni tokens de acceso. Revisamos su contenido antes de compartirlo.

Si todavía no abrimos la reserva o no completamos la revisión humana, el archivo lo deja indicado. Haber descargado el ZIP no significa que esos pasos ya estén terminados. El CSV conserva los decimales sin redondear para poder comprobar los cálculos.

In [24]:

# @title Validación final y exportación de evidencia - RESUA

import json
import hashlib
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import files

OPCION_ELEGIDA = "A_semantica"
OPCIONES_GENERACION = (OPCION_ELEGIDA,)

# --------------------------------------------------
# 1. Pruebas sintéticas de la fusión RRF
# --------------------------------------------------

fusion_prueba, _ = fusionar_rrf([{"chunk_id": x} for x in ["A", "B", "C"]],[{"chunk_id": x} for x in ["C", "A", "D"]])
assert [r["chunk_id"] for r in fusion_prueba] == ["A", "C", "B", "D"]
assert np.isclose(fusion_prueba[0]["puntaje_rrf"],1 / 61 + 1 / 62)
assert fusion_prueba[2]["aporte_bm25"] == 0
assert fusion_prueba[3]["aporte_semantica"] == 0

# --------------------------------------------------
# 2. Pruebas sintéticas de las métricas de recuperación
# --------------------------------------------------
prueba = metricas([{"chunk_id": "C"}, {"chunk_id": "A"}, {"chunk_id": "D"}],{"primera condición": ["A"],"segunda condición": ["B"]})
assert np.isclose(prueba["Precision@3"], 1 / 3)
assert prueba["Recall@3"] == 0.5
assert prueba["RR"] == 0.5
assert prueba["cobertura"] == 0.5
prueba_mejora = metricas([{"chunk_id": "A"}, {"chunk_id": "B"}, {"chunk_id": "E"}],{"primera condición": ["A"],"segunda condición": ["B"]})
assert np.isclose(prueba_mejora["Precision@3"], 2 / 3)
assert prueba_mejora["Recall@3"] == 1
assert prueba_mejora["RR"] == 1
assert prueba_mejora["cobertura"] == 1

assert metricas([], {"condición": ["A"]})["RR"] == 0
assert metricas([], {})["Recall@3"] is None

print("Pruebas sintéticas de RRF y métricas: correctas.")

# --------------------------------------------------
# 3. Comprobaciones antes de exportar
# --------------------------------------------------

OPCION_ELEGIDA = "A_semantica"

assert OPCION_ELEGIDA in OPCIONES, ("La estrategia seleccionada no aparece en OPCIONES.")

errores_tecnicos = [f for f in filas_metricas if f.get("error_tecnico")]

if errores_tecnicos:
    print("Se encontraron errores técnicos:")
    display(pd.DataFrame(errores_tecnicos))
    raise RuntimeError("Resuelve o documenta los errores técnicos antes de entregar.")

revisiones_pendientes = {
    opcion: revision
    for opcion, revision in REVISION_MANUAL.items()
    if any(revision.get(campo) is None for campo in ["correcta","fiel","citas_respaldan","estado_adecuado"])
}
if revisiones_pendientes:
    print("Advertencia: hay criterios de revisión humana pendientes. " "Completa la revisión antes de la entrega final.")
    display(pd.DataFrame(revisiones_pendientes).T)

# --------------------------------------------------
# 4. Huella del corpus y configuración experimental
# --------------------------------------------------
huella_fragmentos = hashlib.sha256(json.dumps(fragmentos,sort_keys=True,ensure_ascii=False,default=str).encode("utf-8")).hexdigest()
evidencia_final = {
    "proyecto": ("RESUA - Sistema de Registro de Avistamientos de especies Silvestres en Zonas Urbanas de Armenia, Quindío"),
    "fecha_utc": FECHA,
    "corpus_id": CORPUS_ID,
    "configuracion_embeddings": configuracion,
    "estrategia_elegida": OPCION_ELEGIDA,
    "decision_desarrollo": DECISION_DESARROLLO,
    "opciones_evaluadas": list(OPCIONES),
    "opciones_generacion": list(OPCIONES_GENERACION),
    "reranker": {
        "modelo": MODELO_RERANKER,
        "revision": REVISION_RERANKER,
        "max_length": 512
    },
    "bm25": {
        "biblioteca": "rank-bm25",
        "version": "0.2.2",
        "algoritmo": "BM25Okapi",
        "k1": BM25_K1,
        "b": BM25_B,
        "epsilon": BM25_EPSILON,
        "tokenizacion": ("NFKC + casefold + palabras y códigos con guion; ""sin stemming ni stopwords"),
        "fragmentos_indexados": len(corpus_tokens),"filtro": "al menos un token coincidente"},
        "rrf": {"constante": C_RRF,"pesos": {"semantica": 1, "bm25": 1},"posiciones_desde": 1,"aporte_ausente": 0,"desempate": "chunk_id ascendente"
    },
    "parametros_evaluacion": {"M": M,"K": K},
    "prompt_version": PROMPT_VERSION,
    "prompt_sistema": PROMPT_SISTEMA,
    "modelo_generativo": MODELO_GENERATIVO,
    "temperatura": 0,
    "reasoning_effort": "low",
    "max_completion_tokens": 1536,
    "casos_anotados": casos,
    "reserva_abierta": any(f.get("grupo") == "reserva"for f in filas_metricas),
    "metricas": filas_metricas,
    "recorridos": recorridos,
    "pregunta_generada": CONSULTA,
    "respuestas": respuestas,
    "revision_humana": REVISION_MANUAL,
    "huella_fragmentos_sha256": huella_fragmentos,
    "limites": (
        "Corpus didáctico pequeño. La evaluación de relevancia "
        "depende de las anotaciones existentes. Las pruebas sintéticas "
        "verifican el funcionamiento de RRF y de las métricas, pero "
        "no demuestran por sí solas la calidad de recuperación de RESUA. "
        "Los tiempos corresponden a ejecuciones puntuales."
    )
}

# --------------------------------------------------
# 5. Normalizar tipos para guardar JSON válido
# --------------------------------------------------

def normalizar_json(obj):
    if isinstance(obj, dict):
        return {str(k): normalizar_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [normalizar_json(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return normalizar_json(obj.tolist())
    if isinstance(obj, np.generic):
        return normalizar_json(obj.item())
    if isinstance(obj, float) and not np.isfinite(obj):
        return None
    if obj is pd.NA or obj is pd.NaT:
        return None
    return obj
evidencia_final = normalizar_json(evidencia_final)

# --------------------------------------------------
# 6. Exportar JSON, CSV y ZIP
# --------------------------------------------------

salida_json = Path("RESUA_Evidencia_Evaluacion_RAG.json")
salida_csv = Path("RESUA_Metricas_RAG.csv")
salida_zip = Path("RESUA_Evidencia_Evaluacion_RAG.zip")

salida_json.write_text(json.dumps(evidencia_final,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
pd.DataFrame(filas_metricas).to_csv(salida_csv,index=False,encoding="utf-8-sig")
with zipfile.ZipFile(salida_zip,"w",zipfile.ZIP_DEFLATED) as archivo:
    archivo.write(salida_json, salida_json.name)
    archivo.write(salida_csv, salida_csv.name)
print("\nEvidencia exportada correctamente.")
print("Proyecto:", evidencia_final["proyecto"])
print("Estrategia seleccionada:", OPCION_ELEGIDA)
print("Reserva abierta:", evidencia_final["reserva_abierta"])
print("Revisión humana pendiente:",bool(revisiones_pendientes))
print("JSON:", salida_json.name)
print("CSV:", salida_csv.name)
print("ZIP:", salida_zip.name)
files.download(str(salida_zip))


Pruebas sintéticas de RRF y métricas: correctas.

Evidencia exportada correctamente.
Proyecto: RESUA - Sistema de Registro de Avistamientos de especies Silvestres en Zonas Urbanas de Armenia, Quindío
Estrategia seleccionada: A_semantica
Reserva abierta: False
Revisión humana pendiente: False
JSON: RESUA_Evidencia_Evaluacion_RAG.json
CSV: RESUA_Metricas_RAG.csv
ZIP: RESUA_Evidencia_Evaluacion_RAG.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Trabajo del equipo
Continuaremos el mismo proyecto. Conservaremos el corpus autorizado, su configuración y los permisos de Supabase. Cambiaremos las preguntas y las evidencias del punto 4 por casos reales del proyecto, con sus fuentes. Incluiremos preguntas directas, paráfrasis, condiciones, una consulta ambigua y otra sin cobertura. Cuando corresponda, añadiremos códigos exactos, vigencia y datos que el corpus no permite conocer. Un conjunto de **12 a 20 casos** puede servir como primer ejercicio de clase, sin presentarlo como una certificación.

Compararemos **A, B y C**, manteniendo preguntas, corpus y generador. Primero justificaremos si BM25 + RRF aporta frente a la línea base semántica; después, si el reranker aporta frente a la híbrida. Dejaremos preguntas de reserva sin usar para ajustar y revisaremos manualmente respuestas útiles y fallos. Si empatamos, si empeoramos o si una etapa no aporta, conservaremos esa evidencia y explicaremos la decisión. No cambiaremos la recuperación semántica por `ts_rank`: búsqueda de texto completo y BM25 no son lo mismo.

En la bitácora quedarán el notebook, las fuentes, la anotación, la tabla comparativa, las trazas, la decisión y la limitación que sigue pendiente. Para el segundo corte debemos poder ejecutar el sistema completo y seguir una afirmación hasta su fragmento de respaldo. **La métrica de recuperación no reemplaza la revisión de la respuesta.**

### Referencias de consulta
- Dorian Brown. `rank-bm25`, versión 0.2.2: `BM25Okapi`, parámetros y preprocesamiento del corpus y de las consultas.
- Cormack, Clarke y Büttcher (2009). Reciprocal Rank Fusion. SIGIR: fusión mediante posiciones de rankings.
- Microsoft Learn. Relevance scoring in hybrid search using Reciprocal Rank Fusion (RRF): constante, aportes y separación de los puntajes originales.
- Sentence Transformers. Uso de CrossEncoder y procedimiento Retrieve & Re-Rank.
- Sentence Transformers. Ficha del modelo multilingüe `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`, revisión indicada en el punto 1.
- Supabase. Documentación de Python: RPC, inicio de sesión con contraseña y filtros de consultas. Documentación de permisos con Row Level Security.
- Groq. Structured Outputs y compatibilidad de `openai/gpt-oss-20b` con JSON Schema estricto.
- Pydantic. Validación de modelos y método `model_validate_json`.